# Toxicity Classification: Handling High-Dimensional, Low-Sample Data

This notebook executes a machine learning pipeline to classify compounds as **Toxic** or **NonToxic**.

**The Mathematical Constraint:** The dataset presents an extreme $p \gg n$ problem (1,204 features vs. 171 samples). Standard models will memorize the training data and fail on unseen samples. To force generalization, this pipeline implements L1 regularization for ruthless dimensionality reduction and SMOTE for minority class augmentation.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, make_scorer

# imblearn pipeline is mandatory. Standard sklearn pipelines apply SMOTE to validation folds, which causes data leakage.
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

## 1. Data Ingestion & Encoding

Machine learning models require numerical targets. The target variable `Class` is binary-encoded. The initial class distribution reveals an imbalance that will bias a naive model toward the majority class (NonToxic).

In [3]:
# Load dataset
df = pd.read_csv('data.csv')
X = df.drop(columns=['Class'])
y = df['Class']

# Encode target (NonToxic -> 0, Toxic -> 1)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"Dataset Loaded: {X.shape[0]} rows, {X.shape[1]} features.")
print(f"Class Distribution: {np.bincount(y_encoded)} (0: NonToxic, 1: Toxic)")

Dataset Loaded: 171 rows, 1203 features.
Class Distribution: [115  56] (0: NonToxic, 1: Toxic)


## 2. Pipeline Architecture



Feeding 1,204 variables into a decision tree with 171 rows guarantees catastrophic overfitting. This pipeline operates in three stages:
1. **Standardization:** Normalizes feature scales.
2. **Lasso (L1) Regularization:** A Logistic Regression model applies a strict mathematical penalty (`C=0.1`) to the coefficients of useless features, crushing them to exactly zero.
3. **SMOTE:** Synthesizes realistic minority class samples in the multi-dimensional space to balance the training split.

In [4]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),

    # L1 Feature Selection wrapper
    ('feature_selection', SelectFromModel(LogisticRegression(
        penalty='l1', solver='liblinear', class_weight='balanced', random_state=42
    ))),

    # Synthetic Minority Over-sampling
    ('smote', SMOTE(random_state=42)),

    # Base Ensemble Model
    ('classifier', RandomForestClassifier(random_state=42))
])

## 3. Hyperparameter Optimization & Cross-Validation



Testing on a single holdout set of 34 rows is statistically invalid. We deploy a `RepeatedStratifiedKFold` evaluation framework. The dataset is split into 5 folds, and the entire training process is repeated 3 times. We simultaneously run a Grid Search to find the optimal constraints for the Random Forest to prevent it from memorizing the surviving features.

In [5]:
param_grid = {
    'feature_selection__estimator__C': [0.1, 0.5, 1.0],
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [3, 5],
    'classifier__min_samples_leaf': [2, 4]
}

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scorer = make_scorer(f1_score, average='macro')

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring=scorer,
    n_jobs=-1,
    verbose=1
)

print("Executing optimization matrix...")
grid.fit(X, y_encoded)

Executing optimization matrix...
Fitting 15 folds for each of 24 candidates, totalling 360 fits


GridSearchCV(cv=RepeatedStratifiedKFold(n_repeats=3, n_splits=5, random_state=42),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('feature_selection',
                                        SelectFromModel(estimator=LogisticRegression(class_weight='balanced',
                                                                                     penalty='l1',
                                                                                     random_state=42,
                                                                                     solver='liblinear'))),
                                       ('smote', SMOTE(random_state=42)),
                                       ('classifier',
                                        RandomForestClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'classifier__max_depth': [3, 5],
                         'classifier__min_samples_leaf': [2, 4],
                         'classifier__n_estimators': [100, 200],
                         'feature_selection__estimator__C': [0.1, 0.5, 1.0]},
             scoring=make_scorer(f1_score, response_method='predict', average=macro),
             verbose=1)

## 4. Diagnostics & Feature Extraction

The Cross-Validated Macro F1 Score is the true measure of this model's predictive power on unseen data. The gap between the CV score and the final in-sample report quantifies the remaining overfitting. We also extract the exact features that survived the L1 purge to identify the actual drivers of toxicity.

In [6]:
print("="*40)
print(f"Best CV Macro F1 Score: {grid.best_score_:.3f}")
print("="*40)
print("Optimal Parameters:")
for k, v in grid.best_params_.items():
    print(f" - {k}: {v}")

# Extract surviving features
best_fs = grid.best_estimator_.named_steps['feature_selection']
surviving_mask = best_fs.get_support()
surviving_features = X.columns[surviving_mask]

print(f"\nDimensionality Reduction: {len(surviving_features)} out of {X.shape[1]} features retained.")
print("Top Retained Features:", list(surviving_features[:10]))

# Final Report
y_pred = grid.predict(X)
print("\nFinal Classification Report (In-Sample Training Data):")
print(classification_report(y_encoded, y_pred, target_names=le.classes_))

Best CV Macro F1 Score: 0.519
Optimal Parameters:
 - classifier__max_depth: 3
 - classifier__min_samples_leaf: 2
 - classifier__n_estimators: 200
 - feature_selection__estimator__C: 0.1

Dimensionality Reduction: 23 out of 1203 features retained.
Top Retained Features: ['MATS3p', 'minHBint4', 'VR2_Dt', 'MATS1s', 'maxHtCH', 'nHtCH', 'minssNH', 'ATSC3e', 'khs.tCH', 'ATSC1v']

Final Classification Report (In-Sample Training Data):
              precision    recall  f1-score   support

    NonToxic       0.94      0.72      0.82       115
       Toxic       0.61      0.91      0.73        56

    accuracy                           0.78       171
   macro avg       0.78      0.82      0.78       171
weighted avg       0.84      0.78      0.79       171

